# Laboratorium 4 — segmentacja naczyń siatkówki: pojęcia, które warto znać

Materiał pomocniczy do instrukcji Prof. Hachaja („lab_03 — segmentacja”). Zakłada znajomość Pythona, NumPy i podstaw sieci splotowych; wszystko, co pochodzi z przetwarzania obrazów i medycyny, jest wyjaśnione od zera. Terminy angielskie w nawiasach — tak nazywają się parametry funkcji, z których będziecie korzystać.

## 1. Zbiór danych DRIVE („retina”)

- 40 kolorowych zdjęć **dna oka** (ang. *fundus*), 565×584 px (szerokość × wysokość; w NumPy tablica ma kształt `(584, 565)` = wiersze × kolumny), zrobionych funduskamerą przez źrenicę. Widać na nich: naczynia krwionośne (ciemne, rozgałęzione), **tarczę nerwu wzrokowego** (jasne koło, z którego naczynia „wychodzą”) i **plamkę** (ciemniejszy obszar w środku).
- Podział: `training` (20 obrazów) i `test` (20 obrazów). Dla każdego obrazu są dwie maski binarne:
  - **segmentacja ręczna** (ang. *ground truth*, GT): piksel = 1 tam, gdzie okulista zaznaczył naczynie; to „prawda”, z którą porównujemy wyniki;
  - **maska pola widzenia** (ang. *field of view*, FOV): piksel = 1 wewnątrz koła, na którym w ogóle jest obraz; poza nim zdjęcie jest czarne. **Oceniamy tylko piksele w FOV** — poza nim zadanie jest trywialne (same zera) i zawyżałoby wyniki.
- Liczby, które warto mieć w głowie: ~12–13 % pikseli w FOV to naczynia (reszta to tło) — **silna nierównowaga klas**; naczynia mają szerokość od ~1 do ~12 px.
- W katalogu `test/2nd_manual` leżą pliki `*_manual1.tif` — to jest ground truth dla testu (nazwa katalogu jest myląca).

## 2. Segmentacja = klasyfikacja każdego piksela

Klasyfikacja obrazu odpowiada „co jest na obrazie”, segmentacja — „który piksel należy do czego”. W tym labie są dwie klasy: naczynie (1) / tło (0). Każda z metod zwraca dla piksela **liczbę** (odpowiedź filtru albo prawdopodobieństwo), a maskę binarną dostaje się przez **próg** (ang. *threshold*). Stąd dwa poziomy oceny:

- **bez progu** — krzywa ROC i AUC oceniają, czy liczby *porządkują* piksele poprawnie (naczynia wyżej niż tło);
- **z progiem** — Dice i IoU oceniają gotową maskę.

## 3. Filtr Frangiego (ang. *vesselness filter*) — detektor „rurek”

Metoda bez uczenia (1998), nadal używana jako punkt odniesienia. Intuicja: w przekroju poprzecznym naczynie jest **doliną jasności** (ciemne na jaśniejszym tle), a wzdłuż naczynia jasność jest prawie stała. Jak to wykryć?

1. **Wygładzenie** obrazu filtrem Gaussa o odchyleniu **σ** (`sigmas`) — usuwa szum i ustala, jak grube struktury nas interesują (σ ≈ promień naczynia w pikselach).
2. **Hesjan** (ang. *Hessian*) — macierz 2×2 drugich pochodnych jasności w każdym pikselu: jak jasność „wygina się” w poziomie, w pionie i po skosie.
3. **Wartości własne** hesjanu λ₁, λ₂ (|λ₁| ≤ |λ₂|) — krzywizny w dwóch prostopadłych kierunkach. Obrazowo: przetnij ciemne naczynie w poprzek — profil jasności ma kształt litery U, czyli dużą drugą pochodną; idź wzdłuż naczynia — jasność jest prawie prosta, druga pochodna ≈ 0. Stąd dla rurki λ₁ ≈ 0 i |λ₂| duże; dla plamy obie są duże, dla płaskiego tła obie małe.
4. Dwie cechy: **R_B = |λ₁|/|λ₂|** (0 = linia, 1 = plama) i **S = √(λ₁² + λ₂²)** („ile w ogóle jest struktury”; małe w tle).
5. Odpowiedź: `V = exp(−R_B²/2β²) · (1 − exp(−S²/2γ²))` — duża, gdy coś jest linią (mały R_B) i ma wyraźny kontrast (duże S). Parametry **β** (tolerancja na plamowatość, domyślnie 0,5) i **γ** (próg kontrastu) są stałe. `scikit-image` zawsze przelicza wejście na skalę [0, 1]; w wersji 0.19 (środowisko Prof. Hachaja) domyślne jest `gamma=15`, przez co odpowiedzi są bardzo małe (10⁻⁸–10⁻⁴), od wersji 0.20 (Colab) `gamma` jest dobierane automatycznie i odpowiedzi sięgają ~0,9. Na kolejność pikseli (a więc na AUC) wartość γ prawie nie wpływa — zmienia głównie skalę liczb.
6. **Polaryzacja**: znak λ₂ mówi, czy rurka jest ciemna, czy jasna. `skimage.filters.frangi(..., black_ridges=True)` (domyślnie) szuka **ciemnych** struktur — tak jak naczynia siatkówki. Dla jasnych naczyń (np. angiografia MR) trzeba `black_ridges=False` albo odwrócić obraz.
7. **Wieloskalowość**: filtr liczy się dla kilku σ i dla każdego piksela bierze maksimum po skalach — cienkie naczynia „łapią” małe σ, grube duże. Żeby odpowiedzi z różnych skal były porównywalne, hesjan mnoży się przez σ² (wygładzanie osłabia pochodne tym bardziej, im większe σ). Tak jest w artykule Frangiego i w scikit-image 0.19; **od wersji 0.20 scikit-image tego mnożenia nie robi** — wtedy maksimum faworyzuje najmniejsze σ i te same parametry dają AUC ok. 0,86 zamiast ok. 0,92. Dlatego wynik zadania 1 zależy od wersji biblioteki (komórka 0 notatnika ją wypisuje).

Co trzeba wiedzieć w praktyce: odpowiedź filtru ma **bardzo skośny rozkład** (mediana ≈ 0, maksimum w obrazie to zwykle brzeg pola widzenia, który też wygląda jak gruba rurka) — surowy wynik wyświetlony wprost jest prawie czarny, a próg 0,5 daje pustą maskę. Nie jest to prawdopodobieństwo. Filtr „widzi” też tarczę nerwu wzrokowego, krawędź FOV i tekstury tła, a grube naczynia z jasnym refleksem w środku potrafi rozdwoić.

## 4. U-Net

Sieć splotowa do segmentacji (2015, obrazowanie biomedyczne), w kształcie litery U:

- **enkoder** (ścieżka kontrakcji): bloki [splot 3×3 + ReLU] ×2, po każdym *max-pooling* 2×2 — rozdzielczość maleje (584→292→146→…), liczba map cech rośnie (dzięki splotom, nie poolingowi); sieć uczy się „co” jest na obrazie;
- **bottleneck**: najmniejsza rozdzielczość, najbardziej skondensowana reprezentacja;
- **dekoder** (ścieżka ekspansji): *transposed convolution* (ang. *Conv2DTranspose*, „splot odwrotny” ze skokiem 2 podwaja rozdzielczość) + sploty — odzyskuje „gdzie”;
- **skip connections** (połączenia przeskokowe): mapy z enkodera są sklejane (`concatenate`) z mapami dekodera na tym samym poziomie — przez nie wracają szczegóły położenia krawędzi, które pooling zgubił; to one dają ostre granice;
- **wyjście**: splot 1×1 z sigmoidą → prawdopodobieństwo „naczynie” dla każdego piksela; strata: **binarna entropia krzyżowa** (ang. *binary cross-entropy*, BCE) liczona po pikselach.

Dwie rzeczy, które nie są oczywiste:

- **Trening na łatach (ang. *patches*), predykcja na całym obrazie.** Sieć uczy się na losowych wycinkach 64×64 (20 obrazów → tysiące wycinków, dodatkowo odbijanych), bo to daje dużo różnorodnych próbek i mieści się w pamięci. Ponieważ U-Net nie ma warstw gęstych (jest *w pełni splotowy*), po treningu można mu podać cały obraz 584×565 — pod warunkiem, że wymiary są **podzielne przez 16** (cztery poolingi ×2), inaczej mapy z enkodera i dekodera nie pasują rozmiarem przy sklejaniu. Rozwiązania: dopełnić obraz do 592×576 i przyciąć wynik, albo (jak w kodzie wykładu) przeskalować do 576×576 i z powrotem.
- **Strata BCE ma „podłogę”**: sieć, która wszędzie przewiduje stałą równą udziałowi naczyń w łatach (ok. 10–12 %), ma stratę ≈ 0,33–0,37. Przez pierwsze 1–2 epoki strata zwykle tam stoi; jeśli stoi przez 4–5 epok, sieć nic nie wykrywa — a nie „dane są trudne”. Dobrze uczący się U-Net schodzi poniżej 0,2 w kilku epokach.

Wersja z **pretrenowanym enkoderem** (ang. *backbone*, np. VGG16 z wagami ImageNet): enkoder zastępuje się siecią klasyfikacyjną uczoną na milionach zdjęć; skip connections biorą jej pośrednie mapy. Czasem pomaga (mało danych), czasem nie (łaty 64×64 siatkówki nie przypominają zdjęć z ImageNet) — na wynikach dołączonych do wykładu (pliki `output_retina_aug_vgg`) wersja z VGG16 wypadła **gorzej** niż od zera (AUC 0,95 vs 0,98).

## 5. Metryki

Dla klasy „naczynie”: **TP** — piksele naczyń wykryte, **FP** — tło uznane za naczynie, **FN** — naczynia pominięte, **TN** — tło uznane za tło.

- **Krzywa ROC** (ang. *receiver operating characteristic*): dla każdego możliwego progu punkt (FPR, TPR), gdzie TPR = TP/(TP+FN) (czułość: jaki odsetek naczyń wykryto), FPR = FP/(FP+TN) (jaki odsetek tła uznano za naczynie). Krzywa bliżej lewego górnego rogu = lepiej. `sklearn.metrics.roc_curve(y_true, y_score)` — oba argumenty to **wektory 1-D** (tu: piksele w FOV ze wszystkich 20 obrazów naraz, ~4,5 mln).
- **AUC** (pole pod krzywą ROC) = prawdopodobieństwo, że losowy piksel naczynia dostanie wyższą odpowiedź niż losowy piksel tła. 0,5 = losowo, 1 = idealnie. **Nie zależy od progu ani od skali odpowiedzi** — można nim porównać filtr (odpowiedź w zupełnie innej skali, np. 10⁻⁶) i sieć (prawdopodobieństwo 0–1). Pułapka: przy 88 % tła AUC jest łaskawe — metoda z AUC 0,95 może dawać kiepskie maski.
- **Dice** = 2·TP / (2·TP + FP + FN) (= F1 dla klasy naczynie) i **IoU** = TP / (TP + FP + FN) (ang. *intersection over union*, indeks Jaccarda): pokrycie maski z ground truth. Ignorują TN, więc nie zawyża ich dominujące tło. Zawsze IoU ≤ Dice (IoU = Dice/(2 − Dice)); obie metryki porządkują metody tak samo. **Wymagają progu** — a wybór progu jest częścią metody: dla sieci 0,5, dla filtru próg trzeba dobrać **na zbiorze treningowym** (dobranie go na testowym to wyciek: ocena na danych, na których coś „ustawiono”).
- **Zbiorczo czy per obraz?** Metryki można liczyć na wszystkich pikselach testu naraz („micro”) albo osobno dla każdego obrazu i uśrednić. Wyniki różnią się nieznacznie; trzeba tylko napisać, co się zrobiło.
- **Accuracy** (odsetek poprawnych pikseli) w tym zadaniu nie mówi nic: „wszystko tło” ma 88 %.

## 6. Pojęcia z Keras, które pojawią się w kodzie

- `layers.Input((None, None, 3))` — sieć przyjmuje obrazy dowolnej wysokości i szerokości (3 kanały RGB); tak trenuje się na łatach, a stosuje na całych obrazach.
- `Conv2DTranspose(n, 2, strides=2)` — podwaja rozdzielczość (dekoder); `MaxPooling2D(2)` — dzieli przez 2 (enkoder); `concatenate([a, b])` — sklejenie map po kanałach (skip connection).
- `model.fit(x, y, batch_size, epochs, validation_split=0.1)` — `x` to łaty `(N, 64, 64, 3)` float32 w [0, 1], `y` maski `(N, 64, 64, 1)` float32 w {0, 1}. Kod wykładu używa `fit_generator` i `Adam(lr=…)` — to API Keras 2, które w środowisku Prof. Hachaja (Keras 2.10) działa; w Keras 3 (Colab) trzeba `fit` i `Adam(learning_rate=…)` (lista zmian jest w notatniku).
- `model.predict(obraz[None])` — sieć zawsze chce wymiaru partii; `[None]` dodaje go, `[0]` zdejmuje z wyniku.
- Trening w tym labie na CPU laptopa trwa kilka–kilkanaście minut (zależnie od liczby epok i rdzeni), na GPU — minuty. GPU w Colabie: *Runtime → Change runtime type → T4 GPU*, **przed** uruchomieniem czegokolwiek (zmiana środowiska restartuje sesję).